In [1]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [2]:
df["sentiment"] = df["sentiment"].apply(lambda x: 1 if x == "positive" else 0)
print(df.head())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


In [3]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=42
)

In [4]:
from collections import Counter
import re

def tokenize(text):
    text = text.lower()
    words = re.findall(r"\b\w+\b", text)
    words = [w for w in words if len(w) > 2]
    return words

print(tokenize(df["review"][0]))

['one', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'episode', 'you', 'hooked', 'they', 'are', 'right', 'this', 'exactly', 'what', 'happened', 'with', 'the', 'first', 'thing', 'that', 'struck', 'about', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'violence', 'which', 'set', 'right', 'from', 'the', 'word', 'trust', 'this', 'not', 'show', 'for', 'the', 'faint', 'hearted', 'timid', 'this', 'show', 'pulls', 'punches', 'with', 'regards', 'drugs', 'sex', 'violence', 'its', 'hardcore', 'the', 'classic', 'use', 'the', 'word', 'called', 'that', 'the', 'nickname', 'given', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focuses', 'mainly', 'emerald', 'city', 'experimental', 'section', 'the', 'prison', 'where', 'all', 'the', 'cells', 'have', 'glass', 'fronts', 'and', 'face', 'inwards', 'privacy', 'not', 'high', 'the', 'agenda', 'city', 'home', 'many', 'aryans', 'muslims', 'gangstas', 'latinos', 'christians', 'italians', 'ir

In [5]:
counter = Counter()

for text in train_texts:
    counter.update(tokenize(text))

vocab_size = 10000

most_common = counter.most_common(vocab_size - 2)

word_to_idx = {word: idx+2 for idx, (word, _) in enumerate(most_common)}

word_to_idx["<PAD>"] = 0
word_to_idx["<OOV>"] = 1

In [6]:
print(list(word_to_idx)[:10])

['the', 'and', 'this', 'that', 'was', 'movie', 'for', 'with', 'but', 'film']


In [7]:
def encode(text):
    tokens = tokenize(text)
    return [word_to_idx.get(word, word_to_idx["<OOV>"]) for word in tokens]

train_sequences = [encode(text) for text in train_texts]
test_sequences = [encode(text) for text in test_texts]

In [8]:
max_length = 256

def pad_truncate(seq):
    if len(seq) > max_length:
        return seq[:max_length]
    else:
        return seq + [0] * (max_length - len(seq))

train_padded = [pad_truncate(seq) for seq in train_sequences]
test_padded = [pad_truncate(seq) for seq in test_sequences]

In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = torch.tensor(train_padded, dtype=torch.long)
y_train = torch.tensor(train_labels.values, dtype=torch.float)

X_test = torch.tensor(test_padded, dtype=torch.long)
y_test = torch.tensor(test_labels.values, dtype=torch.float)

In [10]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [12]:
for batch_x, batch_y in train_loader:
    print(batch_x.shape)
    break

torch.Size([64, 256])


In [13]:
import torch
import torch.nn as nn

class MyRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(MyRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.embedding(x)
        output, _ = self.rnn(x)
        last_output = output[:, -1, :]
        out = self.fc(last_output)
        return self.sigmoid(out)

class MyLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(MyLSTM, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.embedding(x)
        output, _ = self.lstm(x)
        last_output = output[:, -1, :]
        out = self.fc(last_output)
        return self.sigmoid(out)


class MyGRU(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(MyGRU, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.embedding(x)
        output, _ = self.gru(x)
        last_output = output[:, -1, :]
        out = self.fc(last_output)
        return self.sigmoid(out)

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(model, test_loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.long().to(device)
            y_batch = y_batch.float().unsqueeze(1).to(device)

            outputs = model(X_batch)

            preds = (outputs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    return acc, precision, recall, f1, cm

In [21]:
vocab_size = 10000
embed_dim = 100
hidden_dim = 128
lr = 0.001
epochs = 20

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [23]:
models = {
    "RNN": MyRNN(vocab_size, embed_dim, hidden_dim).to(device),
    "LSTM": MyLSTM(vocab_size, embed_dim, hidden_dim).to(device),
    "GRU": MyGRU(vocab_size, embed_dim, hidden_dim).to(device)
}

results = {}

In [24]:
def train_model(model, train_loader, test_loader, lr=0.001, epochs=10):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.long().to(device)
            y_batch = y_batch.float().unsqueeze(1).to(device)

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    acc, precision, recall, f1, cm = evaluate_model(model, test_loader)

    return acc, precision, recall, f1, cm

In [25]:
for name, model in models.items():
    print(f"\nTraining {name} on {device}")
    acc, precision, recall, f1, cm = train_model(model, train_loader, test_loader, epochs=epochs)
    results[name] = (acc, precision, recall, f1, cm)


Training RNN on cuda
Epoch 1/20 - Loss: 0.6949
Epoch 2/20 - Loss: 0.6946
Epoch 3/20 - Loss: 0.6939
Epoch 4/20 - Loss: 0.6934
Epoch 5/20 - Loss: 0.6900
Epoch 6/20 - Loss: 0.6830
Epoch 7/20 - Loss: 0.6778
Epoch 8/20 - Loss: 0.6705
Epoch 9/20 - Loss: 0.6579
Epoch 10/20 - Loss: 0.6481
Epoch 11/20 - Loss: 0.6422
Epoch 12/20 - Loss: 0.6385
Epoch 13/20 - Loss: 0.6281
Epoch 14/20 - Loss: 0.6181
Epoch 15/20 - Loss: 0.6138
Epoch 16/20 - Loss: 0.6036
Epoch 17/20 - Loss: 0.5951
Epoch 18/20 - Loss: 0.5878
Epoch 19/20 - Loss: 0.5813
Epoch 20/20 - Loss: 0.5774

Training LSTM on cuda
Epoch 1/20 - Loss: 0.6903
Epoch 2/20 - Loss: 0.6721
Epoch 3/20 - Loss: 0.6463
Epoch 4/20 - Loss: 0.6405
Epoch 5/20 - Loss: 0.5669
Epoch 6/20 - Loss: 0.5028
Epoch 7/20 - Loss: 0.3334
Epoch 8/20 - Loss: 0.2593
Epoch 9/20 - Loss: 0.2236
Epoch 10/20 - Loss: 0.1849
Epoch 11/20 - Loss: 0.1469
Epoch 12/20 - Loss: 0.1120
Epoch 13/20 - Loss: 0.0840
Epoch 14/20 - Loss: 0.0692
Epoch 15/20 - Loss: 0.0577
Epoch 16/20 - Loss: 0.0473
E

In [26]:
for name, (acc, precision, recall, f1, cm) in results.items():

    print(f"\n{name} Results")
    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("Confusion Matrix:\n", cm)


RNN Results
Accuracy: 0.4919
Precision: 0.49380896226415094
Recall: 0.3324072236554872
F1-score: 0.39734313841774405
Confusion Matrix:
 [[3244 1717]
 [3364 1675]]

LSTM Results
Accuracy: 0.8647
Precision: 0.874289195775792
Recall: 0.8543361778130582
F1-score: 0.8641975308641975
Confusion Matrix:
 [[4342  619]
 [ 734 4305]]

GRU Results
Accuracy: 0.8791
Precision: 0.8935470612412659
Recall: 0.8628696169874975
F1-score: 0.8779404341241797
Confusion Matrix:
 [[4443  518]
 [ 691 4348]]
